In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

In [2]:
X_train = pd.read_csv("X_train_preprocessed.csv")
X_test = pd.read_csv("X_test_preprocessed.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

In [3]:
X_train.shape, X_test.shape

((297649, 211), (74413, 211))

In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [5]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, n_jobs=-1)
log_model.fit(X_train, y_train)

print("Logistic Regression Performance:")
evaluate_model(log_model, X_test, y_test)

Logistic Regression Performance:
Accuracy: 0.6894494241597571
Precision: 0.607078093009652
Recall: 0.3876727680239074
F1 Score: 0.4731790721531973

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.86      0.78     47643
           1       0.61      0.39      0.47     26770

    accuracy                           0.69     74413
   macro avg       0.66      0.62      0.63     74413
weighted avg       0.68      0.69      0.67     74413


Confusion Matrix:
 [[40926  6717]
 [16392 10378]]


In [6]:
from sklearn.tree import DecisionTreeClassifier

In [7]:
dt_gini = DecisionTreeClassifier(criterion="gini", max_depth=10, random_state=42)
dt_gini.fit(X_train, y_train)

print("Decision Tree (Gini) Performance:")
evaluate_model(dt_gini, X_test, y_test)

Decision Tree (Gini) Performance:
Accuracy: 0.6867751602542567
Precision: 0.6018474935278889
Recall: 0.38210683601045947
F1 Score: 0.46744047891056983

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.86      0.78     47643
           1       0.60      0.38      0.47     26770

    accuracy                           0.69     74413
   macro avg       0.66      0.62      0.62     74413
weighted avg       0.67      0.69      0.67     74413


Confusion Matrix:
 [[40876  6767]
 [16541 10229]]


In [8]:
dt_entropy = DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=42)
dt_entropy.fit(X_train, y_train)

print("Decision Tree (Entropy) Performance:")
evaluate_model(dt_entropy, X_test, y_test)

Decision Tree (Entropy) Performance:
Accuracy: 0.6863988819157942
Precision: 0.6065929972684381
Recall: 0.3649981322375794
F1 Score: 0.45575819767713044

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.87      0.78     47643
           1       0.61      0.36      0.46     26770

    accuracy                           0.69     74413
   macro avg       0.66      0.62      0.62     74413
weighted avg       0.67      0.69      0.66     74413


Confusion Matrix:
 [[41306  6337]
 [16999  9771]]


In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [12]:
best_rf = grid_search.best_estimator_

print("Random Forest (Tuned) Performance:")
evaluate_model(best_rf, X_test, y_test)

Random Forest (Tuned) Performance:
Accuracy: 0.6956580167443861
Precision: 0.6012275963663147
Recall: 0.4573776615614494
F1 Score: 0.5195290124111595

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.83      0.78     47643
           1       0.60      0.46      0.52     26770

    accuracy                           0.70     74413
   macro avg       0.67      0.64      0.65     74413
weighted avg       0.68      0.70      0.68     74413


Confusion Matrix:
 [[39522  8121]
 [14526 12244]]


In [13]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (Gini)", "Decision Tree (Entropy)", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, log_model.predict(X_test)),
        accuracy_score(y_test, dt_gini.predict(X_test)),
        accuracy_score(y_test, dt_entropy.predict(X_test)),
        accuracy_score(y_test, best_rf.predict(X_test))
    ],
    "F1 Score": [
        f1_score(y_test, log_model.predict(X_test)),
        f1_score(y_test, dt_gini.predict(X_test)),
        f1_score(y_test, dt_entropy.predict(X_test)),
        f1_score(y_test, best_rf.predict(X_test))
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.689449,0.473179
1,Decision Tree (Gini),0.686775,0.467440
2,Decision Tree (Entropy),0.686399,0.455758
3,Random Forest,0.695658,0.519529


In [15]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_dummy = dummy.predict(X_test)

baseline_f1 = f1_score(y_test, y_dummy)
baseline_f1

0.0

In [16]:
importances = best_rf.feature_importances_
feature_names = X_train.columns

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(15)

,feature,importance
0,usd_goal_real,0.275298
1,duration,0.153314
3,launch_month,0.144454
4,launch_weekday,0.098137
2,launch_year,0.083641
140,category_Tabletop Games,0.009253
172,main_category_Music,0.008656
175,main_category_Technology,0.006228
176,main_category_Theater,0.005827
133,category_Shorts,0.005420
